In [1]:
import os
import struct
import torch
from torch.utils.data import Dataset, DataLoader

In [2]:
class CustomDataset(Dataset):
    def __init__ (self, root, train = True, transform = None):
        self.root = root
        self.train = train
        self.transform = transform

        self.classes = ["T-shirt/top","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankle boot"]

        if train:
            image_file = "train-images-idx3-ubyte"
            label_file = "train-labels-idx1-ubyte"
        else:
            image_file = "t10k-images-idx3-ubyte"
            label_file = "t10k-labels-idx1-ubyte"

        img_path = os.path.join(root, image_file)
        lable_path = os.path.join(root, label_file)

        self.images = self._load_images(img_path)
        self.labels = self._load_lables(lable_path)

    def _load_images(self, img_path):
        with open(img_path, "rb") as f:
            magic, no_of_sample, rows, cols = struct.unpack(">IIII", f.read(16))
            data = f.read()

        images = torch.tensor(list(data), dtype = torch.uint8)
        images = images.reshape(no_of_sample, rows, cols)

        return images

    def _load_lables(self, lable_path):
        with open(lable_path, "rb") as f:
            magic, no_of_lables = struct.unpack(">II", f.read(8))
            data = f.read()

        lables = torch.tensor(list(data), dtype = torch.long)

        return lables

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = self.images[index]
        lable = self.labels[index]

        if self.transform is not None:
            image = self.transform(image)

        return image, lable

In [3]:
root = './data'

In [4]:
root = os.path.join(root, 'FashionMNIST', 'raw')

In [5]:
root

'./data\\FashionMNIST\\raw'

In [6]:
from torchvision import transforms

In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandAugment(num_ops = 2, magnitude = 9, num_magnitude_bins = 31 ),
    transforms.ToTensor()
])

In [8]:
train_data = CustomDataset(
    root,
    train = True,
    transform = transform
)


test_data = CustomDataset(
    root,
    train = False,
)

In [9]:
len(train_data), len(test_data)

(60000, 10000)

In [10]:
img, lable = train_data[0]
iii, lll = test_data[0]
print(f'train sample info : {img.shape}, {lable}, {train_data.classes[lable.item()]}')
print(f'test sample info : {iii.shape}, {lll}, {train_data.classes[lll.item()]}')

train sample info : torch.Size([1, 28, 28]), 9, Ankle boot
test sample info : torch.Size([28, 28]), 9, Ankle boot


In [11]:
class MyCNN(torch.nn.Module):
    def __init__(self, in_channels, no_of_kernels, no_of_classes):
        super().__init__()

        self.block1 = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels = in_channels, out_channels = no_of_kernels, kernel_size = 3, stride = 1, padding = 1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels = no_of_kernels, out_channels = no_of_kernels, kernel_size = 3, stride = 1, padding = 1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size = 2),
            torch.nn.Dropout2d( p = 0.2)
        )

        self.block2 = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels = no_of_kernels, out_channels = no_of_kernels, kernel_size = 3, stride = 1, padding = 1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size = 2),
            torch.nn.Dropout2d( p = 0.2)
        )

        self.classifierblock = torch.nn.Sequential(
            torch.flatten(),
            torch.nn.Dropout(p = 0.5),
            torch.nn.Linear(in_features = no_of_kernels * 7 * 7, out_features = no_of_classes)
        )

    def forward(self, x):
        return(self.classifierblock(self.block2(self.block1(x))))

In [13]:
from torchmetrics import Accuracy

In [14]:
def training_loop(model: torch.nn.Module, data: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, optimizer: torch.optim.Optimizer, accuracy: Accuracy, device: torch.device):
    model = model.to(device)
    accuracy = accuracy.to(device)

    model.train()

    accuracy.reset()
    train_loss = 0

    for X, y in data:
        X, y = X.to(device), y.to(device)

        y_pred = model(X)

        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        accuracy.update(y_pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    acc = accuracy.compute()
    train_loss /= len(data)
    print(f'\ntrain loss: \t{train_loss:.4f} \t||\t train acc: \t{acc:.4f}')

In [15]:
def testing_loop(model: torch.nn.Module, data: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, accuracy: Accuracy, device: torch.device):
    model = model.to(device)
    accuracy = accuracy.to(device)

    model.eval()

    accuracy.reset()
    test_loss = 0

    with torch.inference_mode():
        for X, y in data:
            X, y = X.to(device), y.to(device)

            y_pred = model(X)
            
            loss = loss_fn(y_pred, y)
            test_loss += loss.item()
            accuracy.update(y_pred, y)
  
    acc = accuracy.compute()
    test_loss /= len(data)
    print(f'\ntest loss: \t{test_loss:.4f} \t||\t test acc: \t{acc:.4f}')